# HW1：CIFAR10 速度與準確率挑戰

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/HW1-CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

CIFAR10 是十個類別的 32x32 彩色圖片資料集（飛機、汽車、鳥、貓、鹿、狗、青蛙、馬、船、卡車）。

https://www.cs.toronto.edu/~kriz/cifar.html

**目標：在 Colab T4 上，10 分鐘以內的訓練時間，test accuracy 越高越好。**

規則
* 只能用 CIFAR10 的訓練資料訓練，**不可以用 test set 訓練或調參數**（可以自己從 train 切出 validation）
* 不可以使用預訓練模型（從頭訓練）
* 訓練時間（下面 `train()` 的執行時間）不超過 10 分鐘（T4）
* 繳交：notebook（含執行結果）

可以嘗試的方向
* 模型架構：更寬/更深的 ResNet、不同的 normalization 和 activation
* 資料增強：random crop、flip、cutout、mixup …
* 最佳化：optimizer、learning rate schedule、batch size、label smoothing、weight decay
* 速度：混合精度 (fp16)、`channels_last`、`torch.compile`、資料放在 GPU 上
* Test time augmentation

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from IPython.display import display
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from torchvision.datasets import CIFAR10
train_raw = CIFAR10("data", train=True, download=True)
test_raw = CIFAR10("data", train=False, download=True)
name_array = np.array("飛機、汽車、鳥、貓、鹿、狗、青蛙、馬、船、卡車".split("、"))

mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
def to_tensor(raw):
    X = torch.tensor(raw.data).permute(0, 3, 1, 2).float().div(255)
    return ((X - mean) / std).to(device), torch.tensor(raw.targets).to(device)
train_X, train_y = to_tensor(train_raw)
test_X, test_y = to_tensor(test_raw)
train_X.shape, test_X.shape

In [ ]:
def showX(X):
    display(Image.fromarray(np.concatenate(list(X), axis=1)).resize((64 * len(X), 64), Image.NEAREST))
showX(train_raw.data[:12])
print(name_array[train_raw.targets[:12]])

## 你的程式碼

請完成 `train()`，回傳訓練好的模型，以及 `predict(model, X)`，回傳每張圖的預測類別。

In [ ]:
def train():
    # TODO
    model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 10)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for epoch in range(5):
        perm = torch.randperm(len(train_X), device=device)
        for i in range(0, len(train_X), 256):
            idx = perm[i:i + 256]
            loss = F.cross_entropy(model(train_X[idx]), train_y[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(epoch, loss.item())
    return model

@torch.no_grad()
def predict(model, X):
    # TODO
    model.eval()
    return torch.cat([model(X[i:i + 1000]).argmax(1) for i in range(0, len(X), 1000)])

## 評分（請勿修改）

In [ ]:
print(torch.cuda.get_device_name() if device == "cuda" else "CPU")
torch.cuda.synchronize()
t0 = time.perf_counter()
model = train()
torch.cuda.synchronize()
train_time = time.perf_counter() - t0
acc = (predict(model, test_X) == test_y).float().mean().item()
print(f"訓練時間 {train_time:.1f} 秒,  test accuracy = {acc:.4f}")

## 評分方式

訓練時間在 10 分鐘以內（超過時間以 0 分計），依 test accuracy 計分：

* 94% 以上：30 分
* 92%：25 分
* 90%：20 分
* 85%：12 分
* 80%：6 分

（中間的分數線性內插）